# Roteiro Prático 02 — Starter Kit Python & Gráficos
**Iniciação Científica e Extensionista — CST Sistemas para a Internet — IFSC Garopaba**

## 👋 Primeira vez no Colab? Leia antes de começar
- Este arquivo é um **notebook**: blocos de **texto** (como este) e blocos de **código** (fundo cinza).
- Para executar um bloco de código, clique nele e pressione **Shift + Enter** (ou clique no botão ▶ à esquerda).
- Execute os blocos **em ordem, de cima para baixo**. Na dúvida: menu **Ambiente de execução → Executar tudo**.
- Primeiro, faça **Arquivo → Salvar uma cópia no Drive** e trabalhe na cópia — assim as suas alterações ficam salvas.
- Os arquivos gerados aparecem no painel 📁 (barra lateral esquerda). **Baixe-os logo**: eles são apagados quando a sessão termina.

## Para usar os **seus** dados
1. No painel 📁, envie o seu arquivo `.csv` (ícone de upload ou arraste o arquivo);
2. Ajuste a célula **CONFIGURAÇÃO** (nome do arquivo e das colunas) — é a única que você precisa alterar;
3. Execute tudo de novo (**Ambiente de execução → Executar tudo**).

Ao final, baixe `grafico_barras.pdf`, `grafico_boxplot.pdf` e `tabela_resultados.tex` e envie-os ao Overleaf (Seção 8 do roteiro em PDF).

## 0. Importar as bibliotecas

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

## Configuração
Única célula que você **precisa** alterar para usar os seus dados.

In [ ]:
ARQUIVO_DADOS = "dados_experimento.csv"   # seu CSV (uma linha por medição)
COLUNA_GRUPO = "portal"                   # coluna que identifica os grupos comparados
COLUNA_METRICA = "acessibilidade"         # coluna numérica a ser analisada
ROTULO_EIXO_Y = "Pontuação de acessibilidade (0–100)"   # sempre com a unidade!
LIMITE_Y = (0, 105)                       # use None para escala automática

# CSV exportado do Google Planilhas/Excel em português costuma usar ";" e vírgula decimal.
# Se for o seu caso, troque para: SEPARADOR = ";" e DECIMAL = ","
SEPARADOR = ","
DECIMAL = "."

# Dados de exemplo (usados se o arquivo acima não existir na pasta)
URL_EXEMPLO = (
    "https://raw.githubusercontent.com/chameoandre/"
    "CST-Sistemas-para-internet-iniciacao-cientifica-extensionista/main/"
    "2026-2/roteiro-2-python-graficos/dados_experimento.csv"
)

# Largura de UMA coluna no template ACM sigconf ≈ 3,33 polegadas (8,45 cm).
# Gerar a figura já nesse tamanho garante que as fontes fiquem legíveis no artigo.
LARGURA_COLUNA_POL = 3.33
ALTURA_POL = 2.3

# Paleta amigável a daltônicos (Okabe–Ito)
CORES = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9",
         "#D55E00", "#F0E442", "#000000"]

plt.rcParams.update({
    "font.family": "serif",      # combina com o texto do artigo
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,          # fontes incorporadas corretamente no PDF
})

## 1. Carregar os dados
Se o seu CSV não estiver na pasta, o notebook usa os dados de exemplo da disciplina.

In [ ]:
if os.path.exists(ARQUIVO_DADOS):
    origem = ARQUIVO_DADOS
else:
    origem = URL_EXEMPLO
    print(f"[aviso] '{ARQUIVO_DADOS}' não encontrado — usando os dados de exemplo da disciplina.")

df = pd.read_csv(origem, sep=SEPARADOR, decimal=DECIMAL)
print(f"Dados carregados: {len(df)} linhas, colunas = {list(df.columns)}")
print(df.head(), "\n")

## 2. Estatísticas descritivas por grupo
n, média, desvio padrão, mínimo e máximo de cada grupo.

In [ ]:
resumo = (
    df.groupby(COLUNA_GRUPO)[COLUNA_METRICA]
      .agg(n="count", media="mean", desvio="std", minimo="min", maximo="max")
      .reset_index()
)
media_geral = df[COLUNA_METRICA].mean()

print("Estatísticas descritivas por grupo:")
print(resumo.round(2).to_string(index=False))
print(f"\nMédia geral: {media_geral:.2f} | Desvio padrão geral: {df[COLUNA_METRICA].std():.2f}\n")

grupos = resumo[COLUNA_GRUPO].tolist()
cores = [CORES[i % len(CORES)] for i in range(len(grupos))]

## 3. Gráfico de barras (média ± desvio padrão)
Gerado no tamanho de **uma coluna** do template ACM, para as fontes ficarem legíveis no artigo.

In [ ]:
fig, ax = plt.subplots(figsize=(LARGURA_COLUNA_POL, ALTURA_POL))
ax.bar(grupos, resumo["media"], yerr=resumo["desvio"], capsize=3,
       color=cores, edgecolor="black", linewidth=0.5)
ax.axhline(media_geral, color="#444444", linestyle="--", linewidth=0.8,
           label=f"Média geral ({media_geral:.1f})".replace(".", ","))
ax.set_ylabel(ROTULO_EIXO_Y)
if LIMITE_Y:
    ax.set_ylim(*LIMITE_Y)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False)  # legenda acima, sem cobrir barras
# Sem título dentro da figura: no artigo, quem faz esse papel é o \caption{}.
fig.tight_layout()
fig.savefig("grafico_barras.pdf", bbox_inches="tight")            # vetorial (preferido)
fig.savefig("grafico_barras.png", dpi=300, bbox_inches="tight")   # raster 300 dpi
plt.show()

## 4. Boxplot
Mostra a distribuição (mediana, quartis e valores atípicos) das medições de cada grupo.

In [ ]:
dados_por_grupo = [df.loc[df[COLUNA_GRUPO] == g, COLUNA_METRICA] for g in grupos]
fig, ax = plt.subplots(figsize=(LARGURA_COLUNA_POL, ALTURA_POL))
caixas = ax.boxplot(dados_por_grupo, patch_artist=True, widths=0.6,
                    medianprops={"color": "black", "linewidth": 1})
for caixa, cor in zip(caixas["boxes"], cores):
    caixa.set_facecolor(cor)
    caixa.set_alpha(0.85)
ax.set_xticks(range(1, len(grupos) + 1))
ax.set_xticklabels(grupos)
ax.set_ylabel(ROTULO_EIXO_Y)
# No boxplot deixamos a escala automática: começar em 0 "achataria" as caixas.
fig.tight_layout()
fig.savefig("grafico_boxplot.pdf", bbox_inches="tight")
fig.savefig("grafico_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Tabela LaTeX (booktabs)
Gera `tabela_resultados.tex`, que entra no artigo com `\input{tabela_resultados}`.

In [ ]:
def formatar(valor):
    """Número com 2 casas e vírgula decimal (padrão em português)."""
    return f"{valor:.2f}".replace(".", ",")

linhas = [
    r"\begin{table}[htbp]",
    rf"  \caption{{Estatísticas descritivas de {COLUNA_METRICA} por {COLUNA_GRUPO}}}",
    r"  \label{tab:resultados}",
    r"  \centering",
    r"  \begin{tabular}{lrrrrr}",
    r"    \toprule",
    rf"    \textbf{{{COLUNA_GRUPO.capitalize()}}} & \textbf{{n}} & \textbf{{Média}} & "
    r"\textbf{DP} & \textbf{Mín.} & \textbf{Máx.} \\",
    r"    \midrule",
]
for _, r in resumo.iterrows():
    linhas.append(
        f"    {r[COLUNA_GRUPO]} & {int(r['n'])} & {formatar(r['media'])} & "
        f"{formatar(r['desvio'])} & {formatar(r['minimo'])} & {formatar(r['maximo'])} \\\\"
    )
linhas += [r"    \bottomrule", r"  \end{tabular}", r"\end{table}"]

with open("tabela_resultados.tex", "w", encoding="utf-8") as f:
    f.write("\n".join(linhas) + "\n")

print("-> Arquivos gerados: grafico_barras.pdf/.png, grafico_boxplot.pdf/.png e tabela_resultados.tex")
print("-> Faça o upload deles no Overleaf (Roteiro 02, Seção 8).")